### Test Pydantic AI with LangChain's SQL tools

In [21]:
import os

import nest_asyncio
from dotenv import load_dotenv

load_dotenv("../.env")
neon_conn_string = os.getenv("NEON_DB_URL")
langchain_neon_conn_string = neon_conn_string.replace("postgresql", "postgresql+psycopg")
nest_asyncio.apply() # for async issues in Jupyter Notebook

gpt_model = "gpt-5-mini"

In [4]:
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: ]8;id=184799;https://logfire-us.pydantic.dev/iellis02/blue-horizon\https://logfire-us.pydantic.dev/iellis02/blue-horizon]8;;\

Get the enums

In [5]:
import psycopg

enums = ["availability_status_type", "room_bed_type", "room_status_type", "room_type"]
enum_values = {}

with psycopg.connect(neon_conn_string) as neon_conn:
    with neon_conn.cursor() as cur:
        for enum in enums:
            cur.execute(f"SELECT enum_range(NULL::{enum})")
            enum_values[enum] = [val.strip('\'"{}') for val in cur.fetchall()[0][0].split(',')]
        cur.execute("SELECT DISTINCT unnest(basic_amenities) FROM rooms;")
        basic_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(additional_amenities) FROM rooms;")
        additional_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(view_type) FROM rooms;")
        view_types = [val[0] for val in cur.fetchall()]


print(f"enums = {enum_values}")
print(f"basic amenities = {basic_amenities}")
print(f"additional amenities = {additional_amenities}")
print(f"view types = {view_types}")

enums = {'availability_status_type': ['Booked', 'Available', 'Maintenance'], 'room_bed_type': ['Queen', 'Double Queen', 'King', 'Double King', 'King + Sofa Bed', 'King + Multiple Sofa Beds'], 'room_status_type': ['Available', 'Occupied', 'Maintenance'], 'room_type': ['Standard', 'Deluxe', 'Suite', 'Presidential Suite']}
basic amenities = ['Full Kitchen', 'Executive Office', 'Nespresso Machine', 'Premium Coffee Maker', 'High-Speed WiFi', 'Premium Bathrobes', 'Kitchenette', 'Bathrobes', "Butler's Pantry", 'Welcome Amenity', 'Full-Size Refrigerator', '55" Smart TV', 'Bluetooth Speaker', 'Professional Coffee Bar', 'Guest Bathroom', 'Air Conditioning', 'Living Room', 'Luxury Welcome Amenity', 'Bang & Olufsen Sound System', 'Ultra-High-Speed WiFi', 'Work Desk', 'Multiple 75" Smart TVs', 'In-Room Safe', 'Smart TV', 'Living Room Area', 'Personalized Stationery', 'Hair Dryer', 'Walk-in Closet', 'Multiple Bathrooms', 'Dining Area', 'Bose Sound System', '65" Smart TV', 'Slippers', 'Evening Turndo

Write out the database schema and setup the connection for the LLM.

In [6]:
from langchain_community.utilities import SQLDatabase

schema_description = {
    "rooms": (f"""
            CREATE TABLE rooms (
                room_id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_number INT NOT NULL,
                floor INT NOT NULL,
                type room_type, -- Enum with options {enum_values['room_type']}
                square_feet INT,
                basic_amenities TEXT[],  -- Options are {basic_amenities}
                additional_amenities TEXT[], -- Options are {additional_amenities}
                max_occupancy INT,
                bed_type room_bed_type,  -- Enum with options {enum_values['room_bed_type']}
                view_type TEXT[],  -- Options are {view_types}
                accessibility BOOLEAN,  -- Whether handicapped accessible
                status room_status_type, -- Enum with options {enum_values['room_status_type']}, do not return
                last_renovation DATE, -- Do not provide unless asked for
                base_rate NUMERIC(10, 2),  -- Do not provide unless asked for
                max_rate NUMERIC(10, 2)  -- Do not provide unless asked for
                -- The table lists details about all the rooms in the hotel.
            );
    """),
    "room_availability": (f"""
            CREATE TABLE room_availability (
                id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_id INT NOT NULL, -- Do not return
                room_number INT NOT NULL,
                date DATE NOT NULL,
                status availability_status_type,  -- Enum with options {enum_values['availability_status_type']}
                price NUMERIC(8,2),
                max_occupancy INT,
                FOREIGN KEY (room_id) REFERENCES rooms(room_id)
                -- The table lists the room availability by date and the corresponding rate
            );
    """),
}

db = SQLDatabase.from_uri(database_uri=langchain_neon_conn_string, include_tables=schema_description.keys(), custom_table_info=schema_description)

Setup the LangChain database toolkit

In [22]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI
from pydantic_ai.ext.langchain import LangChainToolset

# 1. Initialize your Language Model (LLM)
# Ensure your LLM supports Pydantic AI/Function calling (e.g., OpenAI, Gemini)
llm = ChatOpenAI(model=gpt_model, temperature=0)

# 2. Initialize the LangChain SQL Toolkit
# This toolkit will use the 'db' object with your custom schema info.
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# 3. Wrap the LangChain Toolkit into a Pydantic AI Toolset
sql_toolset = LangChainToolset(toolkit.get_tools())

Setup the system prompt

In [25]:
dialect="PostgreSQL"
top_k=5

system_prompt = f"""system:
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.

You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
The ONLY exception is chaging a room from "available" to "booked" on specific dates when
making a reservation (booking a room) or from "booked" to "available" when canceling a
reservation. Only make a reservation (book a room) or cancel a reservation when the user
explicitly says to do so.

To start you should ALWAYS look at the tables in the database to see what you can query.

Do NOT skip this step.

Then you should query the schema of the most relevant tables.

If, after querying the schema, anything about the user's query is unclear, ask for
clarification and stop.

If a date is mentioned without a year, assume that the year is 2025.

If asked for multiple consecutive nights, find them by using "GROUP BY" on the room
number and counting the nights using "HAVING COUNT(*)."

Note that, when interacting with the user, a date range specifies from the date of
check-in to the date of check-out. So, January 1-3, 2025 would only be a stay of two
nights, and you would only query the database for January 1-2, 2025. To save the user
some confusion, always state the number of nights when you state a date range.

If you receive an empty result from the SQL query, that is acceptable. Simply use that
result and state that you couldn't find any relevant rooms.

After providing an answer to the user's question, do not offer to do anything that does
not involve querying the database or booking a room.
"""

print(system_prompt)

system:
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct PostgreSQL query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most 5 results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.

You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
The ONLY exception is chaging a room from "available" to "booked" 

Specify the return data format

In [9]:
from pydantic import BaseModel, Field


class Room(BaseModel):
    number: int = Field(..., description="the room number")
    type: str = Field(..., description="the type of room")
    max_occupancy: int = Field(..., description="the maximum number of people allowed to stay in the room")
    bed_type: str = Field(..., description="the types of beds in the room")
    accessibility: bool = Field(description="whether the room is handicapped accessible")
    view_type: list[str] = Field(description="the type of view out the window of the room")
    date: str = Field(description="the date the listing is for")
    price: float = Field(description="the price to book the room for the day")

class Rooms(BaseModel):
    rooms: list[Room] = Field(..., description="details of the rooms")

In [26]:
from pydantic_ai import Agent
from pydantic_ai.settings import ModelSettings

model_settings = ModelSettings(temperature=0)

# Create the Pydantic AI Agent
sql_agent = Agent(
    "openai:" + gpt_model,
    model_settings=model_settings,
    toolsets=[sql_toolset],
    # output_type=Rooms,
    system_prompt=system_prompt,
)

In [143]:
prompt = 'Show me rooms with a view of the ocean and a 55" TV'
result = await sql_agent.run(prompt)

17:40:50.482 sql_agent run
17:40:50.485   chat gpt-5-mini
17:40:52.895   running 1 tool
17:40:52.896     running tool: sql_db_list_tables
17:40:52.899   chat gpt-5-mini
17:40:54.055   running 1 tool
17:40:54.056     running tool: sql_db_schema
17:40:54.060   chat gpt-5-mini
17:41:02.702   running 1 tool
17:41:02.703     running tool: sql_db_query_checker
17:41:11.491   chat gpt-5-mini
17:41:13.953   running 1 tool
17:41:13.954     running tool: sql_db_query
17:41:14.041   chat gpt-5-mini


In [144]:
print(result.output)

Here are up to 5 rooms that have an ocean view and a 55" Smart TV:

1. Room 502 — Floor 5
   - Type: Deluxe
   - Bed: Double Queen
   - Max occupancy: 3
   - Basic amenities: Air Conditioning; 55" Smart TV; Nespresso Machine; Mini Fridge; Hair Dryer; In-Room Safe; Work Desk; High-Speed WiFi; Bluetooth Speaker; Microwave; Premium Bathrobes; Designer Slippers; Evening Turndown Service
   - Additional amenities: Soaking Tub
   - View: Pool View, Ocean View
   - Accessible: No

2. Room 503 — Floor 5
   - Type: Deluxe
   - Bed: Double Queen
   - Max occupancy: 3
   - Basic amenities: Air Conditioning; 55" Smart TV; Nespresso Machine; Mini Fridge; Hair Dryer; In-Room Safe; Work Desk; High-Speed WiFi; Bluetooth Speaker; Microwave; Premium Bathrobes; Designer Slippers; Evening Turndown Service
   - Additional amenities: Soaking Tub; Lounge Access; Balcony
   - View: Pool View, Ocean View
   - Accessible: No

3. Room 504 — Floor 5
   - Type: Deluxe
   - Bed: Double Queen
   - Max occupancy: 3
 

In [13]:
prompt = "I'd like a room for May 20, 2025 for less than $500 with a view of the city."
result = await sql_agent.run(prompt)

13:41:24.002 sql_agent run
13:41:24.004   chat gpt-5.1
13:41:25.001   running 1 tool
13:41:25.001     running tool: sql_db_list_tables
13:41:25.004   chat gpt-5.1
13:41:25.853   running 1 tool
13:41:25.854     running tool: sql_db_schema
13:41:25.858   chat gpt-5.1
13:41:27.553   running 1 tool
13:41:27.554     running tool: sql_db_query_checker
13:41:28.871   chat gpt-5.1
13:41:30.225   running 1 tool
13:41:30.226     running tool: sql_db_query
13:41:30.346   chat gpt-5.1


In [14]:
print(result.output)

Here are rooms available on May 20, 2025 (1 night) under $500 with a city view:

1) Room 112  
- Type: Standard  
- Bed: Queen  
- View: City View / Courtyard View  
- Max occupancy: 2  
- Price: $326.17 (for the night of May 20)

2) Room 319  
- Type: Standard  
- Bed: Queen  
- View: City View / Courtyard View  
- Max occupancy: 2  
- Price: $328.46  

3) Room 220  
- Type: Standard  
- Bed: Double Queen  
- View: City View / Courtyard View  
- Max occupancy: 2  
- Price: $328.77  

4) Room 302  
- Type: Standard  
- Bed: Double Queen  
- View: City View / Courtyard View  
- Max occupancy: 2  
- Price: $331.88  

5) Room 305  
- Type: Standard  
- Bed: Double Queen  
- View: City View  
- Max occupancy: 2  
- Price: $339.14  

Tell me which room number you’d like, and I can book it for May 20, 2025.


In [27]:
prompt = "Does the hotel have any rooms with a cold plunge pool and a view of the Eiffel Tower?"
result = await sql_agent.run(prompt)

13:50:04.296 sql_agent run
13:50:04.298   chat gpt-5-mini
13:50:07.581   running 1 tool
13:50:07.582     running tool: sql_db_list_tables
13:50:07.584   chat gpt-5-mini
13:50:09.432   running 1 tool
13:50:09.433     running tool: sql_db_schema
13:50:09.437   chat gpt-5-mini
13:50:27.033   running 1 tool
13:50:27.034     running tool: sql_db_query_checker
13:50:39.797   chat gpt-5-mini
13:50:42.582   running 1 tool
13:50:42.583     running tool: sql_db_query
13:50:42.669   chat gpt-5-mini
13:50:52.731   running 1 tool
13:50:52.731     running tool: sql_db_query_checker
13:51:05.046   chat gpt-5-mini
13:51:07.188   running 1 tool
13:51:07.188     running tool: sql_db_query
13:51:07.276   chat gpt-5-mini


In [28]:
print(result.output)

I couldn't find any rooms that explicitly match both "cold plunge" (or "plunge"/"cold") and an "Eiffel" view.

Notes:
- I searched the rooms table for amenities containing "plunge" or "cold" and for view_type containing "Eiffel" and got no matches.
- The rooms.view_type values in the schema are generic (e.g., "City View", "Panoramic Ocean View") — there isn't a specific "Eiffel Tower" view option, so the database may record that as "City View" or another generic value instead of "Eiffel".

Would you like me to:
- Search instead for rooms with "Private Pool" (or "Jacuzzi/Soaking Tub") plus "City View" or "Panoramic" as a proxy for an Eiffel view, or
- Search for any rooms that mention "Private Pool" regardless of view, or
- Do something else (please clarify what you mean by "cold plunge")?

Tell me which option you prefer and I’ll run the query (showing up to 5 matching rooms).


In [29]:
prompt = "How much to book the presidential suite for the week of January 5th?"
result = await sql_agent.run(prompt)

13:52:17.123 sql_agent run
13:52:17.125   chat gpt-5-mini
13:52:18.265   running 1 tool
13:52:18.266     running tool: sql_db_list_tables
13:52:18.269   chat gpt-5-mini
13:52:19.758   running 1 tool
13:52:19.759     running tool: sql_db_schema
13:52:19.763   chat gpt-5-mini
13:52:32.403   running 1 tool
13:52:32.404     running tool: sql_db_query_checker
13:53:03.087   chat gpt-5-mini
13:53:05.256   running 1 tool
13:53:05.257     running tool: sql_db_query
13:53:05.371   chat gpt-5-mini


In [30]:
print(result.output)

I assumed you mean the 7-night stay from 2025-01-05 (check-in) to 2025-01-12 (check-out). I looked for Presidential Suite rooms that are Available for all 7 nights and summed the nightly rates. Here are up to 5 cheapest matches (room number — total for 7 nights — average per night):

- Room 1809 — $33,266.02 total (7 nights) — $4,752.29/night  
- Room 1913 — $33,385.83 total (7 nights) — $4,769.40/night  
- Room 1903 — $33,532.52 total (7 nights) — $4,790.36/night  
- Room 1829 — $33,566.12 total (7 nights) — $4,795.16/night  
- Room 2014 — $33,677.50 total (7 nights) — $4,811.07/night

If you want me to book one of these rooms for that week, tell me which room number (or say “cheapest available”) and I’ll complete the reservation.


I'm trying to trip-up the LLM with this prompt. So far, gpt-5-mini gets confused with it, but gpt-5 does not. If I can't figure out a way around the problem, I'll have to use gpt-5.

In [31]:
from pydantic_ai import UsageLimits

prompt = "How much to book the penthouse suite for the week of January 5th?"
result = await sql_agent.run(prompt, usage_limits=UsageLimits(tool_calls_limit=10))

13:54:01.375 sql_agent run
13:54:01.377   chat gpt-5-mini
13:54:04.947   running 1 tool
13:54:04.948     running tool: sql_db_list_tables
13:54:04.962   chat gpt-5-mini
13:54:06.341   running 1 tool
13:54:06.342     running tool: sql_db_schema
13:54:06.346   chat gpt-5-mini


In [32]:
print(result.output)

I need one quick clarification before I check prices.

Do you mean a specific room number when you say "the penthouse suite," or do you mean the hotel's top-level suite (the "Presidential Suite" / penthouse)? If you mean the week of January 5th, I'll assume check-in 2025-01-05 and check-out 2025-01-12 (7 nights). Do you want me to check availability and the total price for those 7 nights, or to actually book it if available?


In [147]:
prompt = "How many floors does the hotel have?"
result = await sql_agent.run(prompt)

17:45:51.845 sql_agent run
17:45:51.847   chat gpt-5-mini
17:45:54.143   running 1 tool
17:45:54.143     running tool: sql_db_list_tables
17:45:54.147   chat gpt-5-mini
17:45:59.542   running 1 tool
17:45:59.543     running tool: sql_db_schema
17:45:59.551   chat gpt-5-mini
17:46:02.869   running 1 tool
17:46:02.871     running tool: sql_db_query_checker
17:46:07.937   chat gpt-5-mini
17:46:09.425   running 1 tool
17:46:09.425     running tool: sql_db_query
17:46:09.511   chat gpt-5-mini


In [148]:
print(result.output)

The hotel has 20 floors.


In [151]:
prompt = "How many rooms are on the bottom floor?"
result = await sql_agent.run(prompt)

17:52:46.305 sql_agent run
17:52:46.307   chat gpt-5-mini
17:52:48.851   running 1 tool
17:52:48.852     running tool: sql_db_list_tables
17:52:48.855   chat gpt-5-mini
17:52:50.764   running 1 tool
17:52:50.766     running tool: sql_db_schema
17:52:50.771   chat gpt-5-mini
17:52:55.144   running 1 tool
17:52:55.145     running tool: sql_db_query_checker
17:53:01.455   chat gpt-5-mini
17:53:03.528   running 1 tool
17:53:03.529     running tool: sql_db_query
17:53:03.532   chat gpt-5-mini
17:53:06.474   running 1 tool
17:53:06.475     running tool: sql_db_query_checker
17:53:12.226   chat gpt-5-mini
17:53:13.887   running 1 tool
17:53:13.888     running tool: sql_db_query
17:53:14.598   chat gpt-5-mini
17:53:16.092   running 1 tool
17:53:16.093     running tool: sql_db_query_checker
17:53:22.229   chat gpt-5-mini
17:53:23.244   running 1 tool
17:53:23.244     running tool: sql_db_query
17:53:23.340   chat gpt-5-mini


In [152]:
print(result.output)

There are 30 rooms on the bottom floor (floor 1).
